# 08 · Time Allocation

### Recap & why now
Seven notebooks in, and the segment durations have been typed in by hand. That loose end
has already caused both problems we have met: Notebook 06's middle segment had **too
much** time and wandered metres off route, and Notebook 07's speed limit was infeasible
because a segment had **too little**.

So the durations are not a detail. They are the part of the problem the QP cannot solve
for you, and this notebook explains why and works around it.

### Learning objectives
1. Explain why segment times cannot go inside the QP.
2. Allocate times by **distance** and by a **trapezoidal-profile** estimate.
3. Use the **scaling laws** to find the fastest legal version of a trajectory.
4. Connect the acceleration limit back to the drone's tilt limit.
5. Refine individual segment times with a simple local search.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

In [ ]:
# === The multi-segment solver from Notebook 06 ===========================

def seg_row(N, seg, t, der):
    """A row of the big constraint matrix that touches only segment `seg`."""
    r = np.zeros(N)
    r[seg*NCOEF:(seg+1)*NCOEF] = deriv_row(NCOEF, t, der)
    return r

def build_cost(times, der=4):
    """Block-diagonal Q: one cost_matrix per segment, stacked along the diagonal."""
    Q = np.zeros((len(times)*NCOEF, len(times)*NCOEF))
    for s, T in enumerate(times):
        Q[s*NCOEF:(s+1)*NCOEF, s*NCOEF:(s+1)*NCOEF] = cost_matrix(NCOEF, T, der)
    return Q

def build_constraints(waypoints, times):
    """Waypoints, rest at both ends, and continuity of velocity/acceleration/jerk at each join."""
    m = len(times); N = m*NCOEF
    rows, vals = [], []
    for s in range(m):                             # Every segment starts and ends on its waypoints.
        rows.append(seg_row(N, s, 0.0, 0));      vals.append(waypoints[s])
        rows.append(seg_row(N, s, times[s], 0)); vals.append(waypoints[s+1])
    for der in (1, 2, 3):                          # At rest, in every sense, at both ends.
        rows.append(seg_row(N, 0, 0.0, der));          vals.append(0.0)
        rows.append(seg_row(N, m-1, times[m-1], der)); vals.append(0.0)
    for s in range(m - 1):                         # The two sides of each join must AGREE...
        for der in (1, 2, 3):
            rows.append(seg_row(N, s, times[s], der) - seg_row(N, s+1, 0.0, der))
            vals.append(0.0)                       # ...but we never say WHAT they agree on.
    return np.array(rows), np.array(vals)

def solve_min_snap_1d(waypoints, times, der=4):
    """Equality-constrained QP, solved through the KKT system. One axis."""
    Q = build_cost(times, der)
    A, b = build_constraints(waypoints, times)
    KKT = np.block([[2*Q, A.T], [A, np.zeros((len(b), len(b)))]])
    sol = np.linalg.solve(KKT, np.concatenate([np.zeros(Q.shape[0]), b]))
    return sol[:Q.shape[0]].reshape(len(times), NCOEF)      # Drop the Lagrange multipliers.

def sample(coeffs, times, t, der=0):
    """Evaluate the piecewise polynomial at global time t."""
    edges = np.concatenate([[0.0], np.cumsum(times)])
    if t <= 0:         return poly_val(coeffs[0], 0.0, der)
    if t >= edges[-1]: return poly_val(coeffs[-1], times[-1], der)
    s = int(np.searchsorted(edges, t, side="right") - 1)
    return poly_val(coeffs[s], t - edges[s], der)

def min_snap_3d(waypoints, times):
    """Solve each axis separately and wrap the result in the ref(t) interface the cascade wants."""
    W = np.asarray(waypoints, float)
    coeffs = [solve_min_snap_1d(W[:, axis], times) for axis in range(3)]
    total = float(np.sum(times))
    def ref(t):
        t = min(max(t, 0.0), total)
        p = np.array([sample(coeffs[a], times, t, 0) for a in range(3)])
        v = np.array([sample(coeffs[a], times, t, 1) for a in range(3)])
        acc = np.array([sample(coeffs[a], times, t, 2) for a in range(3)])
        if t >= total:
            v = np.zeros(3); acc = np.zeros(3)     # Hold position once the trajectory is finished.
        return p, v, acc
    return ref, total, coeffs

ROUTE = np.array([(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5), (2.0, 2.0, 1.5), (2.0, 2.0, 2.5), (0, 0, 1.5)])
DURATIONS = [2.5, 3.0, 3.0, 2.0, 4.0]
g = 9.81
print("solver ready — the standing route has %d waypoints and %d segments, %.1f s total" %
      (len(ROUTE), len(DURATIONS), sum(DURATIONS)))

## 1 · Why the times cannot go inside

Look at where a duration appears in the cost matrix from Notebook 04:

$$Q_{ij} \propto T^{\,i+j-7}$$

With $i, j$ up to 7, that is $T$ raised to powers from 1 to 7. The cost is a high-order
rational function of the durations — nothing like the tidy quadratic-in-$c$ structure that
made everything convex. Treat both $c$ and $T$ as unknowns and you get local minima, where
the solver's answer depends on where it started.

The standard split: choose the times **outside** the QP with a heuristic and a feasibility
check, and keep the inside convex.

In [ ]:
print("  duration T     snap cost      peak |a|")
for T_ in (4.0, 2.0, 1.0, 0.5, 0.25):
    C_ = solve_min_snap_1d([0.0, 2.0], [T_])
    Q_ = cost_matrix(NCOEF, T_, 4)
    grid = np.linspace(0, T_, 400)
    print("  %9.2f s %13.1f %13.2f m/s^2" %
          (T_, C_[0] @ Q_ @ C_[0], max(abs(poly_val(C_[0], t_, 2)) for t_ in grid)))

print("\nHalving the time multiplies the snap cost by about 128 = 2^7, and the acceleration by 4.")
print("Those exponents are the powers of T in the cost matrix. A cost that behaves like this is")
print("not something you can hand to a convex solver alongside the coefficients.")

## 2 · Two heuristics

**Proportional to distance.** Give each segment a share of the total time equal to its
share of the path length. One line, no parameters.

**Trapezoidal profile.** A better estimate that knows about the vehicle: pretend the drone
accelerates at $a_{\max}$ up to $v_{\max}$, cruises, then decelerates.

$$T_{\text{seg}} = \begin{cases}
2\sqrt{d/a_{\max}} & \text{if the leg is too short to reach } v_{\max}\\[4pt]
d/v_{\max} + v_{\max}/a_{\max} & \text{otherwise}\end{cases}$$

It is not what the drone will fly — the real trajectory is a smooth polynomial — but as an
*estimate* of how long a leg should take it is good, and it is stated in units the vehicle
cares about.

In [ ]:
LEGS = np.linalg.norm(np.diff(ROUTE, axis=0), axis=1)
print("route legs [m]:", np.round(LEGS, 3), " total path length %.2f m" % LEGS.sum())

def times_proportional(legs, total_time):
    """Split total_time in proportion to each leg's length."""
    legs = np.asarray(legs, float)
    return list(total_time*legs/legs.sum())

def times_trapezoid(legs, v_max, a_max):
    """Estimate each leg's duration from a trapezoidal speed profile."""
    out = []
    for d_ in np.asarray(legs, float):
        if d_ < v_max**2/a_max:                    # Too short to reach v_max: a triangle.
            out.append(2*np.sqrt(d_/a_max))
        else:                                      # Accelerate, cruise, decelerate.
            out.append(d_/v_max + v_max/a_max)
    return out

allocations = {"equal (3 s each)   ": [3.0]*len(LEGS),
               "proportional       ": times_proportional(LEGS, 14.5),
               "trapezoid v1.5 a2.0": times_trapezoid(LEGS, 1.5, 2.0),
               "trapezoid v2.5 a4.0": times_trapezoid(LEGS, 2.5, 4.0)}

print("\n%-21s %7s %-32s %8s %8s %9s" % ("allocation", "total", "segment times [s]", "peak v", "peak a", "tilt"))
for name, times in allocations.items():
    ref, total, _ = min_snap_3d(ROUTE, times)
    grid = np.linspace(0, total, 700)
    V = np.array([ref(t_)[1] for t_ in grid]); A = np.array([ref(t_)[2] for t_ in grid])
    pv, pa = np.linalg.norm(V, axis=1).max(), np.linalg.norm(A, axis=1).max()
    print("%-21s %6.1fs %-32s %8.2f %8.2f %8.1f°" %
          (name, total, np.round(times, 2), pv, pa, np.degrees(np.arctan(pa/g))))

print("\nThe last column is the one that matters. Project 5 capped the commanded tilt at 35°,")
print("which corresponds to %.2f m/s^2 — so a time allocation is really a bet about how hard" % (g*np.tan(np.deg2rad(35))))
print("you are going to ask the vehicle to lean.")

## 3 · The scaling law, and the fastest legal flight

Scaling every segment time by the same factor $\alpha$ changes the derivatives in a
completely predictable way:

$$v \to \frac{v}{\alpha}, \qquad a \to \frac{a}{\alpha^2}, \qquad \text{snap} \to \frac{\text{snap}}{\alpha^4}$$

The **shape of the path does not change at all** — only how fast it is traversed. So peak
acceleration is a monotone function of $\alpha$, and a bisection finds the fastest legal
value in a handful of solves.

In [ ]:
def scale_check(base_times, alpha):
    """Peak speed and acceleration after scaling all times by alpha."""
    ref, total, _ = min_snap_3d(ROUTE, [t_*alpha for t_ in base_times])
    grid = np.linspace(0, total, 700)
    V = np.array([ref(t_)[1] for t_ in grid]); A = np.array([ref(t_)[2] for t_ in grid])
    return total, np.linalg.norm(V, axis=1).max(), np.linalg.norm(A, axis=1).max()

base = times_trapezoid(LEGS, 2.5, 4.0)
print("  alpha    total     peak v     peak a    (note the exact 1/alpha^2)")
for alpha in (0.5, 1.0, 2.0, 4.0):
    total, pv, pa = scale_check(base, alpha)
    print("  %5.2f %8.2fs %9.2f %10.2f" % (alpha, total, pv, pa))

def fastest_alpha(base_times, a_limit, lo=0.2, hi=5.0, iters=28):
    """Smallest uniform scaling whose peak acceleration stays within a_limit."""
    for _ in range(iters):
        mid = 0.5*(lo + hi)
        if scale_check(base_times, mid)[2] <= a_limit: hi = mid
        else:                                         lo = mid
    return hi

a_limit = g*np.tan(np.deg2rad(35))
alpha_star = fastest_alpha(base, a_limit)
total, pv, pa = scale_check(base, alpha_star)
print("\nfastest scaling within a peak acceleration of %.2f m/s^2 (= 35° of tilt):" % a_limit)
print("  alpha %.3f -> total %.2f s, peak speed %.2f m/s, peak acceleration %.2f m/s^2" %
      (alpha_star, total, pv, pa))
print("  That is the whole %.2f m route in under six seconds, down from 14.5 s." % LEGS.sum())

## 4 · Does the drone agree?

The planner's answer is a claim, and Project 5 gave us the machinery to check it. The
trajectory above demands **exactly** the acceleration corresponding to the controller's
35° limit — so if we fly it, the drone should sit right at its limit and no further.

That is the theory. The result is not quite what the planner promised, and the reason is
worth understanding.

In [ ]:
# === The vehicle and cascade from Project 5, condensed ===================
def quat_normalize(q): q = np.asarray(q, float); return q/np.linalg.norm(q)
def quat_multiply(a, b):
    aw, ax, ay, az = a; bw, bx, by, bz = b
    return np.array([aw*bw-ax*bx-ay*by-az*bz, aw*bx+ax*bw+ay*bz-az*by,
                     aw*by-ax*bz+ay*bw+az*bx, aw*bz+ax*by-ay*bx+az*bw])
def quat_conjugate(q): return np.array([q[0], -q[1], -q[2], -q[3]])
def quat_to_rotmat(q):
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z), 2*(x*y-w*z), 2*(x*z+w*y)],
                     [2*(x*y+w*z), 1-2*(x*x+z*z), 2*(y*z-w*x)],
                     [2*(x*z-w*y), 2*(y*z+w*x), 1-2*(x*x+y*y)]])
def quat_from_rotmat(R):
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr+1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0+R[0,0]-R[1,1]-R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0+R[1,1]-R[0,0]-R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0+R[2,2]-R[0,0]-R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)
def quat_to_euler(q):
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x+y*z), 1-2*(x*x+y*y)), np.arcsin(np.clip(2*(w*y-z*x), -1, 1)),
                     np.arctan2(2*(w*z+x*y), 1-2*(y*y+z*z))])

PARAMS = dict(m=1.0, L=0.25, I=np.diag([0.01, 0.01, 0.02]), d=0.016, T_min=0.0, T_max=6.0)
ARM = PARAMS["L"]/np.sqrt(2)
MOTOR_POS = np.array([[ARM,-ARM,0.], [ARM,ARM,0.], [-ARM,ARM,0.], [-ARM,-ARM,0.]])
SPIN = np.array([-1., 1., -1., 1.])
MIX = np.vstack([np.ones(4), MOTOR_POS[:,1], -MOTOR_POS[:,0], -SPIN*PARAMS["d"]])
POS, VEL, QUAT, OMEGA = slice(0,3), slice(3,6), slice(6,10), slice(10,13)
GAINS = dict(Kp=np.array([2.,2,3]), Kv=np.array([4.,4,5]), K_R=np.array([12.,12,6]),
             K_w=np.array([.06,.06,.06]), v_max=4.0, tilt_max=np.deg2rad(35))

def quad_dynamics(s, T4, p=PARAMS, f_ext=np.zeros(3)):
    q = quat_normalize(s[QUAT]); w = s[OMEGA]
    T, tx, ty, tz = MIX @ np.asarray(T4, float)
    v_dot = (quat_to_rotmat(q) @ np.array([0,0,T]) + np.array([0,0,-p["m"]*g]) + f_ext)/p["m"]
    return np.concatenate([s[VEL], v_dot, 0.5*quat_multiply(q, np.array([0., *w])),
                           np.linalg.solve(p["I"], np.array([tx,ty,tz]) - np.cross(w, p["I"] @ w))])

def rk4_step(s, T4, dt, p=PARAMS, f_ext=np.zeros(3)):
    k1 = quad_dynamics(s, T4, p, f_ext); k2 = quad_dynamics(s+dt/2*k1, T4, p, f_ext)
    k3 = quad_dynamics(s+dt/2*k2, T4, p, f_ext); k4 = quad_dynamics(s+dt*k3, T4, p, f_ext)
    s2 = s + dt/6*(k1+2*k2+2*k3+k4); s2[QUAT] = quat_normalize(s2[QUAT]); return s2

def cascade(s, p_des, v_ff, a_ff, yaw=0.0, p=PARAMS, K=GAINS):
    """Project 5's six controllers, condensed into one function."""
    v_cmd = K["Kp"]*(p_des - s[POS]) + v_ff
    n_ = np.linalg.norm(v_cmd)
    if n_ > K["v_max"]: v_cmd = v_cmd*K["v_max"]/n_
    a_cmd = K["Kv"]*(v_cmd - s[VEL]) + a_ff
    F = p["m"]*(a_cmd + np.array([0, 0, g])); fz = max(F[2], 0.4*p["m"]*g); fxy = F[:2]
    mx = fz*np.tan(K["tilt_max"])
    if np.linalg.norm(fxy) > mx: fxy = fxy*mx/np.linalg.norm(fxy)
    F = np.array([fxy[0], fxy[1], fz]); T = float(F @ quat_to_rotmat(s[QUAT])[:, 2])
    z_des = F/np.linalg.norm(F); x_c = np.array([np.cos(yaw), np.sin(yaw), 0.])
    y_des = np.cross(z_des, x_c); y_des /= np.linalg.norm(y_des)
    q_des = quat_from_rotmat(np.column_stack([np.cross(y_des, z_des), y_des, z_des]))
    q_e = quat_multiply(quat_conjugate(s[QUAT]), q_des)
    if q_e[0] < 0: q_e = -q_e
    tau = K["K_w"]*(2*K["K_R"]*q_e[1:] - s[OMEGA]) + np.cross(s[OMEGA], p["I"] @ s[OMEGA])
    return np.clip(np.linalg.solve(MIX, np.array([T, *tau])), p["T_min"], p["T_max"])

def fly(ref, T_end, dt=0.005, p=PARAMS, K=GAINS, f_ext=lambda t: np.zeros(3)):
    """Closed-loop flight through the Project 5 cascade."""
    s = np.concatenate([[0,0,0], [0,0,0], [1,0,0,0], [0,0,0]]).astype(float)
    ts, xs, ms, rs = [0.0], [s.copy()], [], []
    for k in range(int(round(T_end/dt))):
        p_des, v_ff, a_ff = ref(k*dt)
        T4 = cascade(s, p_des, v_ff, a_ff, 0.0, p, K)
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4); rs.append(p_des)
    return np.array(ts), np.array(xs), np.array(ms), np.array(rs)

def profile_peaks(ref, total, n=900):
    """Peak speed and peak acceleration a trajectory demands."""
    grid = np.linspace(0, total, n)
    V = np.array([ref(t_)[1] for t_ in grid]); A = np.array([ref(t_)[2] for t_ in grid])
    return np.linalg.norm(V, axis=1).max(), np.linalg.norm(A, axis=1).max()

print("vehicle + cascade loaded — Project 5, unchanged")

In [ ]:
print("%-26s %7s %11s %11s %10s" % ("allocation", "total", "RMS error", "peak tilt", "saturated"))
for name, times in [("proportional (14.5 s)", times_proportional(LEGS, 14.5)),
                    ("trapezoid v1.5 a2.0  ", times_trapezoid(LEGS, 1.5, 2.0)),
                    ("bisected to the limit", [t_*alpha_star for t_ in base])]:
    ref, total, _ = min_snap_3d(ROUTE, times)
    t, X, M, R = fly(ref, total + 3.0)
    err = np.linalg.norm(X[1:, POS] - R, axis=1)
    rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))
    print("%-26s %6.1fs %10.4f m %10.1f° %9.1f%%" %
          (name, total, np.sqrt(np.mean(err**2)), np.abs(rpy[:, :2]).max(),
           100*np.mean(M.max(axis=1) >= PARAMS["T_max"]-1e-9)))

print("\nRead the last row carefully. The planner sized that trajectory for exactly 35° of tilt,")
print("and the drone flew it at rather more. That is not a bug in either piece: the planner's")
print("acceleration is the FEEDFORWARD demand — the tilt needed if the drone were perfectly on")
print("the path. It never is, so the feedback loops add a correction on top, and corrections cost")
print("extra tilt. Plan right at the boundary and the correction has nowhere to go.")

In [ ]:
print("planning margin sweep — the fraction of the 35° limit the PLANNER is allowed to use:")
print("  fraction   planned a    total    RMS error   actual peak tilt")
for frac in (1.0, 0.85, 0.7, 0.55):
    alpha_f = fastest_alpha(base, frac*a_limit)
    ref, total, _ = min_snap_3d(ROUTE, [t_*alpha_f for t_ in base])
    t, X, M, R = fly(ref, total + 3.0)
    err = np.linalg.norm(X[1:, POS] - R, axis=1)
    rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))
    print("  %7.0f%% %10.2f %8.2fs %11.4f m %14.1f°" %
          (100*frac, frac*a_limit, total, np.sqrt(np.mean(err**2)), np.abs(rpy[:, :2]).max()))

print("\nThe actual tilt always overshoots the planned one, and the gap grows as you plan closer")
print("to the limit — a more aggressive trajectory produces larger tracking errors, which need")
print("larger corrections. Around 70%% of the limit the drone stays inside 35°, and the flight is")
print("still far quicker than 14.5 s. That fraction is a property of THIS controller and THIS")
print("vehicle: measure it, do not inherit it from a textbook.")

## 🧪 Try it yourself

**E1.** Why does scaling all times by $\alpha$ leave the path unchanged but divide
acceleration by $\alpha^2$? Which quantity divides by $\alpha^4$?

**E2.** The trapezoidal estimate assumes the drone stops at every waypoint, but our
trajectory does not. Why is it still useful — and what does the bisection in Section 3
remove?

In [ ]:
# --- Solution E1 ---
_, pv1, pa1 = scale_check(base, 1.0)
_, pv2, pa2 = scale_check(base, 2.0)
print("E1: doubling all times -> peak v ratio %.3f (expect 0.5), peak a ratio %.3f (expect 0.25)" %
      (pv2/pv1, pa2/pa1))
print("    The path is the same set of POINTS; only the clock changes. Each time derivative brings")
print("    one factor of 1/alpha, so velocity gets 1/alpha, acceleration 1/alpha^2, jerk 1/alpha^3")
print("    and SNAP 1/alpha^4. That last one is why flying slower is so dramatically kinder to the")
print("    motors: half the speed is one sixteenth of the snap.")

# --- Solution E2 ---
print("\nE2: it is an estimate of the right ORDER, and order is what matters when allocating time —")
print("    a leg twice as long needs roughly twice as long whether or not the drone stops. The")
print("    stop-at-waypoint assumption makes it conservative, since the real trajectory carries")
print("    speed through the joins and could go faster. The bisection removes exactly that")
print("    conservatism by scaling the whole thing until a real limit binds.")
prop = times_proportional(LEGS, 14.5)
print("\n    A demonstration of what slack time does — stretching one segment fourfold:")
for label, times in [("proportional        ", prop),
                     ("segment 3 stretched ", [prop[0], prop[1], prop[2]*4, prop[3], prop[4]])]:
    ref, total, _ = min_snap_3d(ROUTE, times)
    grid = np.linspace(0, total, 600)
    P = np.array([ref(t_)[0] for t_ in grid])
    strayed = max(min(np.linalg.norm(p_ - (a_ + np.clip((p_-a_) @ (b_-a_)/max((b_-a_) @ (b_-a_), 1e-9), 0, 1)*(b_-a_)))
                      for a_, b_ in zip(ROUTE[:-1], ROUTE[1:])) for p_ in P)
    pv, pa = profile_peaks(ref, total)
    print("    %s total %5.1fs, peak v %.2f, furthest from the route %.2f m" %
          (label, total, pv, strayed))
print("    The over-long segment does not slow down — it WANDERS. Slack time is not idle time to")
print("    an optimiser: it is room to make a smoother curve, and a smoother curve here means a")
print("    bigger detour. Notebook 09 forbids it outright.")

## 🚁 Mini-project: the same route at four speeds

Animate the drone flying the identical geometric path at four different time scalings.
The shape never changes; the tilt demand does, quadratically. Watch the fastest one lean
until it runs out of margin.

In [ ]:
scalings = [2.0, 1.2, 0.8, alpha_star]
runs = []
for alpha in scalings:
    ref, total, _ = min_snap_3d(ROUTE, [t_*alpha for t_ in base])
    t, X, M, R = fly(ref, total + 1.5)
    rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))
    runs.append((t, X, rpy, total))

longest = max(len(r[0]) for r in runs)
fig = plt.figure(figsize=(9.8, 4.4))

def frame(j):
    fig.clf()
    ax = fig.add_subplot(121, projection="3d")
    ax2 = fig.add_subplot(122)
    for (t, X, rpy, total), col, alpha in zip(runs, ["C0", "C2", "C1", "C3"], scalings):
        k = min(int(j*len(t)/longest), len(t)-1)
        ax.plot(X[:k+1, 0], X[:k+1, 1], X[:k+1, 2], color=col, lw=1.6, label="%.1fx (%.1f s)" % (alpha, total))
        ax2.plot(t[:k+1], np.abs(rpy[:k+1, :2]).max(axis=1), color=col, lw=1.5)
    ax.plot(ROUTE[:, 0], ROUTE[:, 1], ROUTE[:, 2], "*", color="k", ms=8)
    ax.set_xlim(-0.6, 2.8); ax.set_ylim(-0.6, 2.8); ax.set_zlim(0, 3)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z"); ax.legend(fontsize=7)
    ax.view_init(elev=24, azim=-62)
    ax2.axhline(35, color="C3", ls="--", lw=1.2)
    ax2.set_xlim(0, max(r[0][-1] for r in runs)); ax2.set_ylim(0, 55)
    ax2.set_xlabel("time [s]"); ax2.set_ylabel("tilt [deg]")
    return []

anim = animation.FuncAnimation(fig, frame, frames=90, interval=60, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Time allocation is where trajectory planning stops being
> tidy mathematics. Mellinger and Kumar used gradient descent on the segment times with
> the QP in the inner loop; later work has used backtracking line search, bilevel
> optimisation, and learned initialisations. All of them are outer loops around a convex
> inner problem, and all accept the same bargain: give up global optimality over the
> times, keep exactness over the coefficients.
>
> The margin lesson is equally standard. Every production planner plans to a fraction of
> the vehicle's true limits, because the controller needs authority left over to correct
> what the planner did not predict.

**Where next.** Notebook 06's excursion is still unsolved — sensible times reduce it, but
nothing yet *forbids* the path from leaving the route. Notebook 09 adds obstacles, and two
different ways to avoid them.